# PIZZA semantic parsing: Qwen3.5-0.8B + LoRA, three output formats

Evaluates the cafe-parser approach on the public PIZZA benchmark (Arkoudas et al., 2022).

**RQ1** Can a 0.8B decoder with LoRA match the published BART-Large EXR result (78.56 ± 0.31
unordered EM on test)?
**RQ2** Does a compact DSL keep accuracy while cutting output tokens and latency, versus EXR and JSON?
**RQ3** Does grammar-constrained decoding remove structural errors, and what semantic errors remain?

Exact match always comes from the dataset's own `semantic_matchers.is_unordered_exact_match`;
it is never reimplemented. Dev is used for every selection decision; **test is evaluated once
per final configuration**.

The data is CC-BY-NC-4.0. Nothing trained here may be used by the production cafe parser.


## 0. Pin one GPU

Run before anything imports torch. `Trainer` wraps the model in DataParallel when several GPUs
are visible, which splits the batch; and an out-of-range `CUDA_VISIBLE_DEVICES` yields _zero_
devices and silently falls back to CPU. Both are rejected by `assert_single_gpu()`.


In [ ]:
import os

GPU = "0"
SHARD = (0, 1)  # (index, total): one process per GPU

os.environ["CUDA_VISIBLE_DEVICES"] = GPU
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import sys

assert "torch" not in sys.modules, "torch already imported; restart the kernel"
print(f"pinned to GPU {GPU}, shard {SHARD}")

## 1. Environment and data


In [ ]:
INSTALL = True

if INSTALL:
    %pip install -q unsloth unsloth_zoo
    %pip install -q "transformers>=5" trl datasets peft anytree matplotlib
    %pip install -q causal-conv1d flash-linear-attention   # Qwen3.5 hybrid kernels

REPO = os.environ.get("COFFEE_LLM_REPO", "/content/coffee-llm")
if not os.path.isdir(os.path.join(REPO, "src")):
    for guess in ("..", ".", "/content/coffee-llm"):
        if os.path.isdir(os.path.join(guess, "src/pizza_bench")):
            REPO = os.path.abspath(guess)
            break
    else:
        raise SystemExit("clone the repo, or set COFFEE_LLM_REPO")
sys.path.insert(0, os.path.join(REPO, "src"))
os.chdir(REPO)

PIZZA_DIR = os.path.join(REPO, "data/external/pizza")
if not os.path.isdir(PIZZA_DIR):
    !git clone -q --depth 1 https://github.com/amazon-science/pizza-semantic-parsing-dataset.git {PIZZA_DIR}
os.environ["PIZZA_DIR"] = PIZZA_DIR

In [ ]:
from pizza_bench import data, formats, grammar, metrics, report
from pizza_bench import experiment as X

ENV = X.describe_env()
X.assert_single_gpu()
for k, v in ENV.items():
    print(f"{k:16} {v}")
if not ENV["bf16"]:
    print("\nWARNING: no bf16 -> fp32, expect roughly 5x the wall time.")

## 2. Configuration

Best-practice choices, stated so they can be reported:

- **A generous fixed step budget with dev-based early selection**, not fixed epochs. Each train
  size then trains to convergence, so the curve measures _data_ efficiency rather than how long
  each arm was trained. The selected checkpoint is the best dev EM, and `best_step` is logged.
- `max_new_tokens` from the 99th percentile of target lengths: EXR 128, JSON 128, DSL 48.
  Truncation is logged per run; it should stay at 0.
- Identical LoRA hyperparameters across formats; only the LR is tuned, per format, on dev.


In [ ]:
CFG = X.Config(
    formats=("exr", "json", "dsl"),
    sizes=(1000, 5000, 20000),
    seeds=(0, 1, 2),
    shard=SHARD,
    rank=16,
    alpha=16,
    max_steps=2000,
    batch_size=16,
    warmup_steps=50,
    eval_every=100,
    early_stop_patience=5,
    lr_candidates=(1e-4, 2e-4, 5e-4),
)
print(
    f"{len(CFG.formats) * len(CFG.sizes) * len(CFG.seeds)} grid runs"
    f" + {len(CFG.formats) * len(CFG.lr_candidates)} LR-sweep runs"
)

In [ ]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/pizza-runs"
except Exception:
    OUT = os.path.join(REPO, "pizza-runs")

for sub in ("", "predictions", "adapters", "tables", "figures", "tmp"):
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)
RESULTS = os.path.join(OUT, "results.jsonl")
print("writing to", OUT, "|", len(X.load_results(RESULTS)), "rows already logged")

## 3. Data and format sanity


In [ ]:
SPLITS = {s: data.load(s) for s in ("train", "dev", "test")}
print({k: len(v) for k, v in SPLITS.items()})
print(
    "PCFG-error subset:",
    len(data.pcfg_error_subset(SPLITS["test"])),
    "of",
    len(SPLITS["test"]),
)

cat = formats.Catalog()
print("catalog:", {k: len(v) for k, v in cat.ids.items()})

# Round-trip is a hard gate: if this fails, every downstream number is meaningless.
bad = 0
for r in SPLITS["dev"] + SPLITS["test"]:
    node = formats.parse_exr(r["exr"])
    for fmt in CFG.formats:
        if formats.to_exr(
            formats.decode(formats.encode(node, fmt), fmt)
        ) != formats.to_exr(node):
            bad += 1
assert bad == 0, f"{bad} round-trip failures"
print("round-trip on dev+test: clean")

ex = formats.parse_exr(SPLITS["test"][1]["exr"])
print("\nSRC ", SPLITS["test"][1]["src"])
for fmt in CFG.formats:
    print(f"{fmt.upper():5}", formats.encode(ex, fmt))

In [ ]:
# Output length per format, in Qwen tokens (answers RQ2's token-count half).
import statistics

tok = X.tokenizer()
length_rows = []
for fmt in CFG.formats:
    lens = [
        len(
            tok(
                formats.encode(formats.parse_exr(r["exr"]), fmt),
                add_special_tokens=False,
            )["input_ids"]
        )
        for r in SPLITS["test"]
    ]
    lens.sort()
    length_rows.append(
        {
            "format": fmt,
            "mean": statistics.fmean(lens),
            "median": statistics.median(lens),
            "p99": lens[int(len(lens) * 0.99)],
            "max": max(lens),
            "max_new_tokens": X.MAX_NEW[fmt],
        }
    )
print(report.render(length_rows, floatfmt="{:.1f}"))
report.to_csv(length_rows, os.path.join(OUT, "tables", "output_tokens.csv"))
report.to_latex(
    length_rows,
    os.path.join(OUT, "tables", "output_tokens.tex"),
    "Target length in Qwen3.5 tokens on the PIZZA test set.",
    "tab:tokens",
    floatfmt="{:.1f}",
)

## 4. Condition A — base model, few-shot

No fine-tuning. Five fixed train examples, identical across formats, chosen to cover
pizza+drink, negated topping, complex topping, style, and a drink with volume and container.


In [ ]:
shots = X.pick_fewshot(SPLITS["train"])
for r in shots:
    print(r["src"], "\n   ", formats.to_dsl(formats.parse_exr(r["exr"])), "\n")

In [ ]:
for fmt in CFG.formats:
    row = X.run_fewshot(CFG, SPLITS, ENV, OUT, RESULTS, fmt, split="dev")
    m = row["dev"]
    print(f"{fmt:5} dev EM {m['em']:.4f}  validity {m['parse_validity']:.4f}")

## 5. Learning-rate selection (dev only)

One LR per format at n=5000, seed 0, chosen on dev EM, then frozen for the whole grid.


In [ ]:
CFG.lrs = X.select_lrs(CFG, SPLITS, ENV, OUT, RESULTS)
print(CFG.lrs)

## 6. Condition B — LoRA SFT per format

Resumable: runs already in `results.jsonl` are skipped, so re-running after a disconnect
continues. For several GPUs, start one process per GPU with a different `SHARD`.


In [ ]:
X.run_grid(CFG, SPLITS, ENV, OUT, RESULTS, dry_run=True)

In [ ]:
X.run_grid(CFG, SPLITS, ENV, OUT, RESULTS)

## 7. Test evaluation — once per final configuration

Only after dev selection is complete. One run per format at the largest train size, using the
best seed by dev EM.


In [ ]:
ROWS = report.load_results(RESULTS)
best = {}
for r in ROWS:
    if r.get("condition") == "B_sft" and r["size"] == max(CFG.sizes):
        key = r["format"]
        if key not in best or r["dev"]["em"] > best[key]["dev"]["em"]:
            best[key] = r
print({k: (v["seed"], round(v["dev"]["em"], 4)) for k, v in best.items()})

In [ ]:
for fmt, r in best.items():
    if any(
        x.get("condition") == "C_test" and x["format"] == fmt
        for x in report.load_results(RESULTS)
    ):
        print(f"{fmt}: test already evaluated, skipping")
        continue
    row = X.train_one(
        fmt,
        r["size"],
        r["seed"],
        r["lr"],
        SPLITS,
        CFG,
        ENV,
        OUT,
        eval_test=True,
        save_adapter=True,
    )
    row["condition"] = "C_test"
    X.append_result(RESULTS, row)
    print(
        f"{fmt:5} test EM {row['test']['em']:.4f}  "
        f"PCFG-err {row['test']['em_pcfg_err']:.4f}  validity {row['test']['parse_validity']:.4f}"
    )

## 8. Condition C — grammar-constrained decoding

llama.cpp only: Ollama's API accepts JSON schemas, not GBNF. Merge the best DSL adapter, export
GGUF, then run `scripts/pizza_latency.py` with and without `--grammar` on the Mac.


In [ ]:
gbnf = os.path.join(OUT, "pizza_dsl.gbnf")
grammar.write_gbnf(gbnf)
print(open(gbnf).read()[:600], "...")
print("\nMerge + export the best DSL adapter, then on the Mac:")
print(
    "  python scripts/pizza_latency.py --backend llamacpp --model <gguf> --format dsl"
)
print(
    f"  python scripts/pizza_latency.py --backend llamacpp --model <gguf> --format dsl --grammar {gbnf}"
)

## 9. Results


In [ ]:
ROWS = report.load_results(RESULTS)
sft = [r for r in ROWS if r.get("condition") == "B_sft"]
print(f"{len(ROWS)} rows, {len(sft)} SFT runs")
trunc = [r["run_id"] for r in sft if r["dev"].get("truncated", 0) > 0]
print("runs with truncated outputs:", trunc or "none")

for split in ("dev",):
    t = report.summarize(ROWS, split)
    print(f"\n=== unordered EM, {split} ===")
    print(
        report.render(t, ["format", "size", "n_seeds", "mean", "std", "ci_lo", "ci_hi"])
    )
    report.to_csv(t, os.path.join(OUT, "tables", f"em_{split}.csv"))
    report.to_latex(
        t,
        os.path.join(OUT, "tables", f"em_{split}.tex"),
        f"Unordered exact match on {split}, mean over seeds with 95\\% bootstrap CI.",
        f"tab:em-{split}",
    )

In [ ]:
main = []
for r in ROWS:
    if r.get("condition") != "C_test":
        continue
    m = r["test"]
    main.append(
        {
            "condition": f"SFT {r['format'].upper()}",
            "em_test": m["em"],
            "em_pcfg_err": m["em_pcfg_err"],
            "validity": m["parse_validity"],
            "tokens": next(
                x["mean"] for x in length_rows if x["format"] == r["format"]
            ),
        }
    )
for p in report.PUBLISHED:
    main.append(
        {
            "condition": p["system"] + " (published)",
            "em_test": p["em_test"] / 100,
            "em_pcfg_err": None,
            "validity": None,
            "tokens": None,
        }
    )
print(
    report.render(main, ["condition", "em_test", "em_pcfg_err", "validity", "tokens"])
)
report.to_csv(main, os.path.join(OUT, "tables", "main.csv"))
report.to_latex(
    main,
    os.path.join(OUT, "tables", "main.tex"),
    "PIZZA test results. Published baselines are cited, not re-run.",
    "tab:main",
)

In [ ]:
pairs = []
for a, b in [("dsl", "exr"), ("dsl", "json"), ("json", "exr")]:
    pairs += report.paired(ROWS, "dev", a, b)
for p in pairs:
    p["verdict"] = "within noise" if p["within_noise"] else "separated"
print(
    report.render(
        pairs,
        [
            "a",
            "b",
            "size",
            "n_pairs",
            "mean_diff",
            "sd_diff",
            "cohens_dz",
            "ci_lo",
            "ci_hi",
            "verdict",
        ],
    )
)
report.to_csv(pairs, os.path.join(OUT, "tables", "paired.csv"))

In [ ]:
slot = []
for r in ROWS:
    if r.get("condition") != "C_test":
        continue
    for group, v in r["test"]["prf"].items():
        if v["f1"] is not None:
            slot.append(
                {
                    "format": r["format"],
                    "slot": group,
                    "p": v["p"],
                    "r": v["r"],
                    "f1": v["f1"],
                    "tp": v["tp"],
                    "fp": v["fp"],
                    "fn": v["fn"],
                }
            )
print(report.render(slot, ["format", "slot", "p", "r", "f1", "tp", "fp", "fn"]))
report.to_csv(slot, os.path.join(OUT, "tables", "slot_prf.csv"))

halluc = [
    {
        "format": r["format"],
        **{
            k: r["test"][k]
            for k in (
                "hallucinated_entities_per_utt",
                "hallucinated_numbers_per_utt",
                "extra_suborders",
                "missing_suborders",
                "utts_with_hallucination",
            )
        },
    }
    for r in ROWS
    if r.get("condition") == "C_test"
]
print()
print(report.render(halluc))
report.to_csv(halluc, os.path.join(OUT, "tables", "hallucination.csv"))

In [ ]:
FIGS = {"data_efficiency_dev": report.fig_data_efficiency(ROWS, "dev")}
for name, fig in FIGS.items():
    fig.savefig(os.path.join(OUT, "figures", f"{name}.pdf"), bbox_inches="tight")
    fig.savefig(
        os.path.join(OUT, "figures", f"{name}.png"), bbox_inches="tight", dpi=150
    )
print("wrote", sorted(FIGS))

## 10. Error taxonomy

100 dev errors from the best DSL model, categorised and sampled for the paper.


In [ ]:
import json
from collections import Counter

best_dsl = max(
    (r for r in ROWS if r.get("condition") == "B_sft" and r["format"] == "dsl"),
    key=lambda r: r["dev"]["em"],
)
path = os.path.join(OUT, "predictions", f"{best_dsl['run_id']}.dev.jsonl")
preds = [json.loads(l) for l in open(path)]

errors = []
for p in preds:
    pred_exr = metrics.to_exr_or_none(p["pred"], "dsl")
    if pred_exr and metrics.exact_match(pred_exr, p["gold"]):
        continue
    errors.append((p, report.categorize(pred_exr, p["gold"], cat.ids)))

counts = Counter(c for _, c in errors)
print(f"{len(errors)} errors of {len(preds)} dev utterances")
print(
    report.render(
        [
            {"category": k, "n": v, "share": v / len(errors)}
            for k, v in counts.most_common()
        ]
    )
)

for category in counts:
    sample = next(p for p, c in errors if c == category)
    print(
        f"\n--- {category}\n  SRC  {sample['src']}\n  GOLD {sample['gold']}\n  PRED {sample['pred']}"
    )

report.to_csv(
    [
        {"category": c, "src": p["src"], "gold": p["gold"], "pred": p["pred"]}
        for p, c in errors[:100]
    ],
    os.path.join(OUT, "tables", "error_taxonomy.csv"),
)

## Caveats

1. **Published baselines are cited, not re-run** (PCFG 68.02, BERT 64.08, BART-Large EXR 78.56,
   BART on the PCFG-error subset 59.49). They used the full 2.46M train set; the runs here use
   1k/5k/20k from the 10% split, so a like-for-like claim needs the full-data point.
2. **Train is the 10% split.** Every dev and test entity appears in it, so unseen-entity errors
   only arise at the 1k/5k subsets.
3. **Test is evaluated once per final configuration**; everything else is dev.
4. **The DSL preserves EXR sibling order**, so round-trips are character-identical. The official
   matcher is order-insensitive, so this is stricter than correctness requires but keeps the
   training target deterministic.
5. **Latency is measured on an M1 Mac** via Ollama/llama.cpp, not on the training GPU, and is
   therefore a deployment number rather than a throughput number.
6. **CC-BY-NC-4.0**: neither the data nor any model trained on it may go into the production
   cafe parser, and neither belongs in git.
